# TV Series Genre Classification from Plot Descriptions (NLP)

Predict a show's genre from its text description using classic NLP.

> The Kaggle dataset is called "IMDB" but it is a catalogue of TV series and streaming titles, so this is genre prediction, not movie-review sentiment analysis.

| | |
|---|---|
| **Task** | Multi-class text classification (3 genres) |
| **Data** | Kaggle `payamamanat/imbd-dataset`: 9,957 titles, reduced to 1,658 rows in 3 genres (Drama 587, Animation/Action/Adventure 553, Documentary 518) |
| **Model** | NLTK tokenization + WordNet lemmatization, TF-IDF (1-2 grams, 10k features), Multinomial Naive Bayes |
| **Result** | **81.9% test accuracy** (332 test samples); 95.7% on train |

## Approach
- Kept the three most common single-label genre groups so each class has enough examples.
- Cleaned the descriptions with word tokenization and lemmatization.
- Vectorized with TF-IDF unigrams + bigrams (10,000 features).
- Stratified 80/20 split (`random_state=42`), Multinomial Naive Bayes.

## Results

| Class | Precision | Recall | F1 |
|---|---:|---:|---:|
| Drama | 0.71 | 0.92 | 0.80 |
| Animation / Action / Adventure | 0.90 | 0.85 | 0.87 |
| Documentary | 0.92 | 0.67 | 0.78 |

The main error is Documentary predicted as Drama (30 of 104 documentaries). The model is also clearly overfit (95.7% train vs 81.9% test).

## Limitations & next steps
- The TF-IDF vectorizer is fitted on the whole dataset before the train/test split, which leaks a little vocabulary information. Fit it on the training set only (e.g. in a `Pipeline`).
- Stop-word removal is imported but not applied.
- Some rows have placeholder descriptions such as "Add a Plot"; these should be filtered out.
- Next: try Logistic Regression / linear SVM on the same features, add cross-validation, and compare against sentence embeddings.


In [129]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,roc_auc_score,roc_curve



In [130]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("payamamanat/imbd-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imbd-dataset' dataset.
Path to dataset files: /kaggle/input/imbd-dataset


In [131]:
df = pd.read_csv(path+'/IMBD.csv')
df

,title,year,certificate,duration,genre,rating,description,stars,votes
0,Cobra Kai,(2018– ),TV-14,30 min,"Action, Comedy, Drama",8.5,Decades after their 1984 All Valley Karate Tou...,"['Ralph Macchio, ', 'William Zabka, ', 'Courtn...","177,031"
1,The Crown,(2016– ),TV-MA,58 min,"Biography, Drama, History",8.7,Follows the political rivalries and romance of...,"['Claire Foy, ', 'Olivia Colman, ', 'Imelda St...","199,885"
2,Better Call Saul,(2015–2022),TV-MA,46 min,"Crime, Drama",8.9,The trials and tribulations of criminal lawyer...,"['Bob Odenkirk, ', 'Rhea Seehorn, ', 'Jonathan...","501,384"
3,Devil in Ohio,(2022),TV-MA,356 min,"Drama, Horror, Mystery",5.9,When a psychiatrist shelters a mysterious cult...,"['Emily Deschanel, ', 'Sam Jaeger, ', 'Gerardo...","9,773"
4,Cyberpunk: Edgerunners,(2022– ),TV-MA,24 min,"Animation, Action, Adventure",8.6,A Street Kid trying to survive in a technology...,"['Zach Aguilar, ', 'Kenichiro Ohashi, ', 'Emi ...","15,413"
...,...,...,...,...,...,...,...,...,...
9952,The Imperfects,(2022– ),TV-MA,45 min,"Action, Adventure, Drama",6.3,After an experimental gene therapy turns them ...,"['Morgan Taylor Campbell, ', 'Italia Ricci, ',...","3,130"
9953,The Walking Dead,(2010–2022),TV-MA,44 min,"Drama, Horror, Thriller",8.1,Sheriff Deputy Rick Grimes wakes up from a com...,"['Andrew Lincoln, ', 'Norman Reedus, ', 'Melis...","970,067"
9954,The Crown,(2016– ),TV-MA,58 min,"Biography, Drama, History",8.7,Follows the political rivalries and romance of...,"['Claire Foy, ', 'Olivia Colman, ', 'Imelda St...","199,898"
9955,Supernatural,(2005–2020),TV-14,44 min,"Drama, Fantasy, Horror",8.4,Two brothers follow their father's footsteps a...,"['Jared Padalecki, ', 'Jensen Ackles, ', 'Jim ...","439,601"


In [132]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9957 entries, 0 to 9956
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        9957 non-null   object 
 1   year         9430 non-null   object 
 2   certificate  6504 non-null   object 
 3   duration     7921 non-null   object 
 4   genre        9884 non-null   object 
 5   rating       8784 non-null   float64
 6   description  9957 non-null   object 
 7   stars        9957 non-null   object 
 8   votes        8784 non-null   object 
dtypes: float64(1), object(8)
memory usage: 700.2+ KB


In [133]:
df['genre'].value_counts()


,count
genre,
Comedy,859
Drama,587
"Animation, Action, Adventure",553
Documentary,518
"Comedy, Drama",284
...,...
"Short, Action, Comedy",1
"Biography, Thriller",1
"Drama, History, Music",1


In [134]:
selected_genres = ['Drama', 'Animation, Action, Adventure', 'Documentary']
df = df[df['genre'].isin(selected_genres)]
df

,title,year,certificate,duration,genre,rating,description,stars,votes
4,Cyberpunk: Edgerunners,(2022– ),TV-MA,24 min,"Animation, Action, Adventure",8.6,A Street Kid trying to survive in a technology...,"['Zach Aguilar, ', 'Kenichiro Ohashi, ', 'Emi ...","15,413"
41,Partner Track,(2022– ),TV-MA,60 min,Drama,6.8,"Centers on Ingrid Yun, an idealistic young law...","['Arden Cho, ', 'Alexandra Turshen, ', 'Bradle...","3,840"
50,Attack on Titan,(2013–2023),TV-MA,24 min,"Animation, Action, Adventure",9.0,After his hometown is destroyed and his mother...,"['Josh Grelle, ', 'Yûki Kaji, ', 'Yui Ishikawa...","363,297"
63,Diario de un Gigoló,(2022),TV-MA,50 min,Drama,6.3,A gigolo's life begins to unravel when he beco...,"['Jesús Castro, ', 'Victoria White, ', 'Fabiol...",538
76,Arcane,(2021– ),TV-14,41 min,"Animation, Action, Adventure",9.0,Set in utopian Piltover and the oppressed unde...,"['Hailee Steinfeld, ', 'Kevin Alejandro, ', 'J...","196,697"
...,...,...,...,...,...,...,...,...,...
9810,Explained,(2018– ),TV-MA,NaN,Documentary,7.4,Explained looks at diamonds and how they have ...,"['Cleo Constantine Abram, ', 'Karen Smit, ', '...",304
9813,Oni: Thunder God's Tale,(2022– ),TV-Y7,NaN,"Animation, Action, Adventure",NaN,Add a Plot,"[""Daisuke 'Dice' Tsutsumi"", '| ', ' Stars:'...",NaN
9829,Avatar: The Last Airbender,(2005–2008),TV-Y7,25 min,"Animation, Action, Adventure",8.9,The gang hides out in the Western Air Temple. ...,"['Ethan Spaulding', '| ', ' Stars:', 'Zach ...","3,733"
9873,Pokémon,(1997– ),TV-Y,22 min,"Animation, Action, Adventure",6.7,Arriving in Floaroma Town for Dawn's next Poké...,"['Norihiko Sutô', '| ', ' Stars:', 'Sarah N...",79


In [135]:
x = df['description']
y = df['genre']

In [136]:
y = y.map({'Drama':0,'Animation, Action, Adventure':1,'Documentary':2})

Word correction


In [137]:
import re
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [138]:
import nltk
nltk.download('punkt_tab')
lemmatizer = WordNetLemmatizer()
tokenizer = x.apply(word_tokenize)

x = tokenizer.apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
x = x.apply(lambda x: ' '.join(x))
x

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,description
4,A Street Kid trying to survive in a technology...
41,"Centers on Ingrid Yun , an idealistic young la..."
50,After his hometown is destroyed and his mother...
63,A gigolo 's life begin to unravel when he beco...
76,Set in utopian Piltover and the oppressed unde...
...,...
9810,Explained look at diamond and how they have co...
9813,Add a Plot
9829,The gang hide out in the Western Air Temple . ...
9873,Arriving in Floaroma Town for Dawn 's next Pok...


In [139]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1,2),max_features=10000)
x = tfidf.fit_transform(x).toarray()
x = pd.DataFrame(x)
x


,0,1,2,3,4,5,6,7,8,9,...,9990,9991,9992,9993,9994,9995,9996,9997,9998,9999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1653,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1654,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1656,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [140]:

x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [141]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(x_train,y_train)
y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)

In [142]:
print('accuracy score for train ',accuracy_score(y_train,y_pred_train))
print('classificationreport for train ',classification_report(y_train,y_pred_train))
print('confusion matrix for train ',confusion_matrix(y_train,y_pred_train))


accuracy score for train  0.9570135746606335
classificationreport for train                precision    recall  f1-score   support

           0       0.90      1.00      0.95       470
           1       1.00      0.96      0.98       442
           2       0.99      0.91      0.95       414

    accuracy                           0.96      1326
   macro avg       0.96      0.96      0.96      1326
weighted avg       0.96      0.96      0.96      1326

confusion matrix for train  [[468   0   2]
 [ 14 425   3]
 [ 36   2 376]]


In [143]:
print('accuracy score for test ',accuracy_score(y_test,y_pred_test))
print('classificationreport for test ',classification_report(y_test,y_pred_test))
print('confusion matrix for test ',confusion_matrix(y_test,y_pred_test))

accuracy score for test  0.8192771084337349
classificationreport for test                precision    recall  f1-score   support

           0       0.71      0.92      0.80       117
           1       0.90      0.85      0.87       111
           2       0.92      0.67      0.78       104

    accuracy                           0.82       332
   macro avg       0.85      0.81      0.82       332
weighted avg       0.84      0.82      0.82       332

confusion matrix for test  [[108   6   3]
 [ 14  94   3]
 [ 30   4  70]]
